# Results analysis — EEG-only vs. EEG+ECG seizure prediction

Compares the two feature sets on the **SeizeIT2** dataset (within-subject split).

The **headline metric is the per-subject AUC** (within-patient ranking quality), not the
global/pooled AUC: because the model's probability scale differs per patient, pooling all
subjects together destroys the ranking and makes the global AUC look like chance. The
per-subject AUC is the honest measure of whether the model has real signal.

Inputs (produced by `evaluate_eeg.py` / `evaluate_eeg_ecg.py`):
- `../results/seizure_prediction_<eeg|eeg_ecg>/per_subject.csv` — per-subject metrics
- `../results/seizure_prediction_<eeg|eeg_ecg>/metrics.csv` — global metrics per run

In [ ]:
import csv
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

RESULTS = Path('../results')
PIPELINES = {
    'eeg': RESULTS / 'seizure_prediction_eeg',
    'eeg_ecg': RESULTS / 'seizure_prediction_eeg_ecg',
}
OUT_PNG = RESULTS / 'comparison_eeg_vs_eeg_ecg.png'

In [ ]:
# Load per-subject AUC for each feature set (subject -> AUC, or None if no test positives).
def load_per_subject(feature_set):
    p = PIPELINES[feature_set] / 'per_subject.csv'
    out = {}
    if p.exists():
        with open(p, newline='', encoding='utf-8') as f:
            for r in csv.DictReader(f):
                out[r['subject']] = (float(r['auc']) if r['auc'] not in ('', 'nan') else None)
    else:
        print(f'WARNING: {p} not found - run evaluate_{feature_set}.py first.')
    return out

auc_eeg = load_per_subject('eeg')
auc_ecg = load_per_subject('eeg_ecg')
print(f'eeg: {len(auc_eeg)} subjects | eeg_ecg: {len(auc_ecg)} subjects')

In [ ]:
# Paired per-subject comparison (subjects with an AUC in BOTH feature sets).
subjects = sorted(set(auc_eeg) | set(auc_ecg))
paired = [(s, auc_eeg.get(s), auc_ecg.get(s)) for s in subjects
          if auc_eeg.get(s) is not None and auc_ecg.get(s) is not None]

print('Per-subject within-patient AUC:')
print(f"  {'subject':10}{'EEG':>8}{'EEG+ECG':>10}{'delta':>9}")
for s, a, b in paired:
    print(f'  {s:10}{a:>8.3f}{b:>10.3f}{b - a:>+9.3f}')

e = np.array([a for _, a, _ in paired])
c = np.array([b for _, _, b in paired])
if len(paired):
    print('\nMean per-subject AUC:')
    print(f'  EEG     : {e.mean():.3f} +/- {e.std():.3f}')
    print(f'  EEG+ECG : {c.mean():.3f} +/- {c.std():.3f}')
    verdict = 'ECG helps' if (c - e).mean() > 0 else 'ECG does not help'
    print(f'  Mean delta (EEG+ECG - EEG): {(c - e).mean():+.3f}  ({verdict} on average)')

In [ ]:
# Comparison figure: per-subject AUC, EEG vs EEG+ECG.
x = np.arange(len(paired))
w = 0.4
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w / 2, [a for _, a, _ in paired], w, label='EEG')
ax.bar(x + w / 2, [b for _, _, b in paired], w, label='EEG+ECG')
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1, label='chance (0.5)')
ax.set_xticks(x)
ax.set_xticklabels([s for s, _, _ in paired], rotation=45, ha='right')
ax.set_ylabel('Within-patient AUC')
ax.set_ylim(0, 1)
ax.set_title('Seizure prediction: EEG vs. EEG+ECG (per-subject AUC)')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
fig.tight_layout()
OUT_PNG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT_PNG, dpi=150)
print('Saved', OUT_PNG.resolve())
plt.show()

In [ ]:
# Global metrics (latest run per feature set) - the pooled AUC is contaminated,
# shown for completeness only. F1 is at threshold 0.5.
def latest_global(feature_set):
    p = PIPELINES[feature_set] / 'metrics.csv'
    if not p.exists():
        return None
    with open(p, newline='', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    return rows[-1] if rows else None

print(f"{'feature_set':12}{'F1':>8}{'AUC-ROC':>10}{'AUC-PR':>9}{'train_f1':>10}")
print('-' * 49)
for fs in ('eeg', 'eeg_ecg'):
    r = latest_global(fs)
    if r:
        print(f"{fs:12}{float(r['f1']):>8.3f}{float(r['auc_roc']):>10.3f}"
              f"{float(r['auc_pr']):>9.3f}{float(r['train_f1']):>10.3f}")
    else:
        print(f'{fs:12}  (no metrics.csv yet)')